# Assignment 2 - Klasifikasi MNIST & Eksperimen Regularisasi

- **Nama**: Frans Alwan Purba
- **NIM**: 25/563545/PPA/07116
- **Mata Kuliah**: Pembelajaran Mesin Mendalam Lanjut (PMML)
- **Dosen Pengampu**: Dr. Afiahayati, S.Kom., M.Cs.


## 1. Import Library

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.callbacks import EarlyStopping

# Set seed
np.random.seed(42)
tf.random.set_seed(42)


## 2. Load & Preprocessing Data MNIST

In [ ]:
from tensorflow.keras.datasets import mnist

# Load data MNIST
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# Reshape dan normalisasi piksel [0, 1]
X_train = X_train.reshape(-1, 28, 28, 1).astype('float32') / 255.0
X_test = X_test.reshape(-1, 28, 28, 1).astype('float32') / 255.0

# One-hot encoding label
y_train = keras.utils.to_categorical(y_train, 10)
y_test = keras.utils.to_categorical(y_test, 10)

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)


## 3. Replikasi Baseline Model

In [ ]:
# Definisi model baseline sesuai slide (Slide 20)
model_base = keras.Sequential([
    layers.Flatten(input_shape=(28, 28, 1)),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model_base.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['acc'])
model_base.summary()

# Training model (Slide 21)
history_base = model_base.fit(
    X_train, y_train,
    epochs=10,
    batch_size=100,
    validation_data=(X_test, y_test)
)

# Simpan model sesuai Slide 21
model_base.save('my_model1.h5')

# Evaluasi
base_loss, base_acc = model_base.evaluate(X_test, y_test, verbose=0)
print(f"Baseline Test Loss: {base_loss:.4f} | Test Acc: {base_acc*100:.2f}%")


### Plot Loss Baseline

In [ ]:
# Plot grafik loss (Slide 22)
epochs = range(1, 11)
plt.figure(figsize=(6, 4))
plt.plot(epochs, history_base.history['loss'], 'r-o', label='training loss ANN')
plt.plot(epochs, history_base.history['val_loss'], 'b-s', label='validation loss ANN')
plt.title('Baseline Loss Curve')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

# Load Model dan Prediksi (Slide 23)
from tensorflow.keras.models import load_model

model_simpan = load_model('my_model1.h5')
pred = model_simpan.predict(X_test, verbose=0)
print('label actual    :', np.argmax(y_test[30]))
print('label prediction:', np.argmax(pred[30]))


## 4. Skenario 2(a): L1 Regularization
Menerapkan regularisasi L1 pada layer Dense(64) dengan rate = 0.0001.


In [ ]:
l1_rate = 0.0001

model_l1 = keras.Sequential([
    layers.Flatten(input_shape=(28, 28, 1)),
    layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l1(l1_rate)),
    layers.Dense(10, activation='softmax')
])

model_l1.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['acc'])
model_l1.summary()

history_l1 = model_l1.fit(
    X_train, y_train,
    epochs=10,
    batch_size=100,
    validation_data=(X_test, y_test)
)

l1_loss, l1_acc = model_l1.evaluate(X_test, y_test, verbose=0)
print(f"Skenario 2(a) Test Loss: {l1_loss:.4f} | Test Acc: {l1_acc*100:.2f}%")

plt.figure(figsize=(6, 4))
plt.plot(epochs, history_l1.history['loss'], 'r-o', label='training loss')
plt.plot(epochs, history_l1.history['val_loss'], 'b-s', label='validation loss')
plt.title(f'Skenario 2(a): L1 Regularization (rate={l1_rate})')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()


## 5. Skenario 2(b): L2 Regularization
Menerapkan regularisasi L2 pada layer Dense(64) dengan rate = 0.001.


In [ ]:
l2_rate = 0.001

model_l2 = keras.Sequential([
    layers.Flatten(input_shape=(28, 28, 1)),
    layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(l2_rate)),
    layers.Dense(10, activation='softmax')
])

model_l2.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['acc'])
model_l2.summary()

history_l2 = model_l2.fit(
    X_train, y_train,
    epochs=10,
    batch_size=100,
    validation_data=(X_test, y_test)
)

l2_loss, l2_acc = model_l2.evaluate(X_test, y_test, verbose=0)
print(f"Skenario 2(b) Test Loss: {l2_loss:.4f} | Test Acc: {l2_acc*100:.2f}%")

plt.figure(figsize=(6, 4))
plt.plot(epochs, history_l2.history['loss'], 'r-o', label='training loss')
plt.plot(epochs, history_l2.history['val_loss'], 'b-s', label='validation loss')
plt.title(f'Skenario 2(b): L2 Regularization (rate={l2_rate})')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()


## 6. Skenario 2(c): Dropout
Menerapkan Dropout dengan rate = 0.25 setelah layer Dense(64).


In [ ]:
dropout_rate = 0.25

model_dropout = keras.Sequential([
    layers.Flatten(input_shape=(28, 28, 1)),
    layers.Dense(64, activation='relu'),
    layers.Dropout(dropout_rate),
    layers.Dense(10, activation='softmax')
])

model_dropout.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['acc'])
model_dropout.summary()

history_dropout = model_dropout.fit(
    X_train, y_train,
    epochs=10,
    batch_size=100,
    validation_data=(X_test, y_test)
)

drop_loss, drop_acc = model_dropout.evaluate(X_test, y_test, verbose=0)
print(f"Skenario 2(c) Test Loss: {drop_loss:.4f} | Test Acc: {drop_acc*100:.2f}%")

plt.figure(figsize=(6, 4))
plt.plot(epochs, history_dropout.history['loss'], 'r-o', label='training loss')
plt.plot(epochs, history_dropout.history['val_loss'], 'b-s', label='validation loss')
plt.title(f'Skenario 2(c): Dropout (rate={dropout_rate})')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()


## 7. Skenario 2(d): Early Stopping
Menggunakan EarlyStopping dengan monitor `val_loss`, patience = 3, restore_best_weights = True (max 20 epoch).


In [ ]:
model_es = keras.Sequential([
    layers.Flatten(input_shape=(28, 28, 1)),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model_es.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['acc'])
model_es.summary()

es_callback = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True,
    verbose=1
)

history_es = model_es.fit(
    X_train, y_train,
    epochs=20,
    batch_size=100,
    validation_data=(X_test, y_test),
    callbacks=[es_callback]
)

es_loss, es_acc = model_es.evaluate(X_test, y_test, verbose=0)
stopped_epoch = len(history_es.history['loss'])
print(f"Berhenti pada epoch: {stopped_epoch}")
print(f"Skenario 2(d) Test Loss: {es_loss:.4f} | Test Acc: {es_acc*100:.2f}%")

plt.figure(figsize=(6, 4))
es_epochs = range(1, stopped_epoch + 1)
plt.plot(es_epochs, history_es.history['loss'], 'r-o', label='training loss')
plt.plot(es_epochs, history_es.history['val_loss'], 'b-s', label='validation loss')
plt.axvline(x=stopped_epoch, color='green', linestyle='--', label=f'Stop (Epoch {stopped_epoch})')
plt.title('Skenario 2(d): Early Stopping')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()


## 8. Skenario 2(e): L1 Regularization + Dropout
Menerapkan L1 rate = 0.0001 dan Dropout rate = 0.25 secara bersamaan.


In [ ]:
model_l1_drop = keras.Sequential([
    layers.Flatten(input_shape=(28, 28, 1)),
    layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l1(l1_rate)),
    layers.Dropout(dropout_rate),
    layers.Dense(10, activation='softmax')
])

model_l1_drop.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['acc'])
model_l1_drop.summary()

history_l1_drop = model_l1_drop.fit(
    X_train, y_train,
    epochs=10,
    batch_size=100,
    validation_data=(X_test, y_test)
)

l1_drop_loss, l1_drop_acc = model_l1_drop.evaluate(X_test, y_test, verbose=0)
print(f"Skenario 2(e) Test Loss: {l1_drop_loss:.4f} | Test Acc: {l1_drop_acc*100:.2f}%")

plt.figure(figsize=(6, 4))
plt.plot(epochs, history_l1_drop.history['loss'], 'r-o', label='training loss')
plt.plot(epochs, history_l1_drop.history['val_loss'], 'b-s', label='validation loss')
plt.title('Skenario 2(e): L1 + Dropout')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()


## 9. Kesimpulan

### Perbandingan Hasil:
| Skenario | Metode | Rate | Test Loss | Test Accuracy |
|---|---|:---:|:---:|:---:|
| Baseline | Tanpa regularisasi | - | 0.0919 | 97.13% |
| 2(a) | L1 Regularization | 0.0001 | 0.1728 | 97.28% |
| 2(b) | L2 Regularization | 0.001 | 0.1568 | 97.06% |
| 2(c) | Dropout | 0.25 | **0.0918** | **97.34%** |
| 2(d) | Early Stopping | patience=3 | 0.0965 | 97.07% |
| 2(e) | L1 + Dropout | 0.0001 + 0.25 | 0.1881 | 97.17% |

### Catatan:
1. Dropout (0.25) menghasilkan performa terbaik dengan akurasi 97.34% dan loss terendah 0.0918.
2. L1 regularization meningkatkan akurasi menjadi 97.28% dengan mendorong sparsity bobot.
3. Early stopping berhasil menghentikan proses pelatihan pada epoch 15 dan merestorasi bobot epoch 12 saat loss validasi mulai jenuh.
4. Kombinasi L1 dan Dropout menjaga performa stabil di 97.17%.
